# SHAP Explanation for FakeScope Model

This notebook demonstrates how to use SHAP (SHapley Additive exPlanations) to explain the predictions of the FakeScope DistilBERT model.

In [1]:
import sys
import os

# Add the project root to the path so we can import src
sys.path.append(os.path.abspath('..'))

from src.shap_explain import get_shap_explainer, explain_text
import shap

/Users/enriqueestevezalvarez/Library/Mobile Documents/com~apple~CloudDocs/Final Project/FakeScope/FakeScope/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Initialize Explainer

We load the model and tokenizer and create a SHAP explainer. This might take a moment to load the model.

In [3]:
import sys
if 'src.shap_explain' in sys.modules:
    del sys.modules['src.shap_explain']

from src.shap_explain import get_shap_explainer

explainer = get_shap_explainer()

Device set to use cpu


In [5]:
# Define stopwords
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Download NLTK stopwords if not already downloaded
import nltk
nltk.download('stopwords', quiet=True)

# Combine NLTK and sklearn stopwords
stop_words_nltk = set(stopwords.words('english'))
combined_stopwords = set(ENGLISH_STOP_WORDS) | stop_words_nltk

# Add custom domain stopwords
custom_stopwords = {
    'reuters', 'factbox', 'says', 'associated', 'said', 'read', 'press', 'ap', 
    'reporting', 'editing', 'featured image', 'featured', 'image', 'pic twitter', 
    'https', 'twitter com', 'com', 'getty', 'monday', 'tuesday', 'wednesday', 
    'thursday', 'friday', 'saturday', 'sunday'
}
combined_stopwords = combined_stopwords | custom_stopwords

# Define other TF-IDF parameters
TOKEN_PATTERN = r'(?u)\b\w\w+\b'
MIN_DF = 5
MAX_DF = 0.9
NGRAM_RANGE = (1, 2)

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df_news = pd.read_csv('../data/news_cleaned.csv')  # Adjust path as needed

# Prepare X and y
X = df_news['clean_text']
y = df_news['class']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

FileNotFoundError: [Errno 2] No such file or directory: '../data/news_cleaned.csv'

In [ ]:
# TF-IDF Vectorization
# Transform text data into numerical features for baseline models
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer with optimized settings for faster training
vectorizer = TfidfVectorizer(
    max_features=3000,  # ⚡ Reduced from 5000 for faster training
    stop_words=list(combined_stopwords),
    token_pattern=TOKEN_PATTERN,
    min_df=MIN_DF,      # min_df=5 (remove rare tokens)
    max_df=MAX_DF,      # max_df=0.9 (remove very common tokens)
    ngram_range=NGRAM_RANGE,
)

# Fit on training data and transform both train and test
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"✅ TF-IDF vectorization complete (OPTIMIZED)")
print(f"   Train matrix shape: {X_train_tfidf.shape}")
print(f"   Test matrix shape: {X_test_tfidf.shape}")
print(f"   Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"   max_features=3000 (reduced for faster training)")

# Save vectorizer for later use
from joblib import dump
dump(vectorizer, 'tfidf_vectorizer.joblib')
print(f"✅ Vectorizer saved to tfidf_vectorizer.joblib")

NameError: name 'X_train' is not defined

In [2]:
# Simple SHAP Explanation - No data loading needed!
import sys
if 'src.shap_explain' in sys.modules:
    del sys.modules['src.shap_explain']

from src.shap_explain import get_shap_explainer, explain_text

# Load the explainer (this loads the model)
explainer = get_shap_explainer()

# Define some sample texts to explain
sample_texts = [
    "Says before he planned a rally on June 19 nobody had ever heard of Juneteenth",
    "Says these elite figures are on house arrest with ankle monitors due to child trafficking crimes."
]

# Explain each text
for i, text in enumerate(sample_texts):
    print(f"\n{'='*60}")
    print(f"Sample {i+1}: {text[:50]}...")
    print(f"{'='*60}")
    
    # Get SHAP explanation
    shap_values = explain_text(text, explainer)
    
    # Visualize (this will show which words contribute to the prediction)
    shap.plots.text(shap_values)

Device set to use cpu



Sample 1: Says before he planned a rally on June 19 nobody h...



Sample 2: Says these elite figures are on house arrest with ...


In [ ]:
explainer = get_shap_explainer()

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './models/distilbert_fakenews_2stage'. Use `repo_type` argument if needed.

## Explain Predictions

We will now explain the model's prediction for a sample text. 
The model predicts whether a news article is **Fake** or **True**.

In [ ]:
# Sample Fake News Text
fake_text = "Breaking: Aliens have landed in New York City and are demanding to speak with the President immediately. Witnesses say they arrived in a giant silver saucer."

# Sample True News Text
true_text = "The stock market closed higher today as investors reacted positively to the latest jobs report. The Dow Jones Industrial Average rose by 200 points."

texts = [fake_text, true_text]

In [ ]:
shap_values = explain_text(texts, explainer)

NameError: name 'explainer' is not defined

## Visualize Explanations

The plot below shows how each word contributes to the prediction. 
- **Red** indicates features that push the prediction towards the 'True' class (or whatever the positive class is).
- **Blue** indicates features that push the prediction towards the 'Fake' class.

*(Note: Check the output class labels to confirm direction)*

In [9]:
shap.plots.text(shap_values)